In [13]:
import re

def unify(x, y, subs=None):
    if subs is None:
        subs = {}
    if x == y:
        return subs
    if isinstance(x, str) and x.startswith('?'):
        subs[x] = y
        return subs
    if isinstance(y, str) and y.startswith('?'):
        subs[y] = x
        return subs
    return None

def literal(neg, predicate, *args):
    return (neg, predicate, tuple(args))

def display_literal(lit):
    neg, pred, args = lit
    return ("~" if neg else "") + pred + "(" + ", ".join(args) + ")"

def display_clause(clause):
    if not clause:
        return "{}"
    return " OR ".join(display_literal(x) for x in clause)

def convert_to_fol(sentence):
    s = sentence.lower().strip().rstrip(".")
    if "likes all kind" in s:
        return (
            "FOR ALL x: Food(x) -> Likes(John,x)",
            [[literal(True, "Food", "?x"),
              literal(False, "Likes", "john", "?x")]]
        )
    if "and" in s and "are food" in s:
        return (
            "Food(Apple) AND Food(Vegetables)",
            [
                [literal(False, "Food", "apple")],
                [literal(False, "Food", "vegetables")]
            ]
        )
    if "eats" in s and "not killed" in s and "food" in s:
        return (
            "FOR ALL x,y: (Eats(y,x) AND NotKilled(y)) -> Food(x)",
            [[
                literal(True, "Eats", "?y", "?x"),
                literal(True, "NotKilled", "?y"),
                literal(False, "Food", "?x")
            ]]
        )
    if "eats" in s and "alive" in s:
        match = re.search(r"(\w+) eats (\w+)", s)
        if match:
            person, item = match.groups()
            return (
                f"Eats({person},{item}) AND Alive({person})",
                [
                    [literal(False, "Eats", person, item)],
                    [literal(False, "Alive", person)]
                ]
            )
    if "everything that" in s:
        match = re.search(
            r"(\w+) eats everything that (\w+) eats", s
        )
        if match:
            person1, person2 = match.groups()
            return (
                f"FOR ALL x: Eats({person2},x) -> Eats({person1},x)",
                [[
                    literal(True, "Eats", person2, "?x"),
                    literal(False, "Eats", person1, "?x")
                ]]
            )
    if "alive implies not killed" in s:
        return (
            "FOR ALL x: Alive(x) -> NotKilled(x)",
            [[
                literal(True, "Alive", "?x"),
                literal(False, "NotKilled", "?x")
            ]]
        )
    if "not killed implies alive" in s:
        return (
            "FOR ALL x: NotKilled(x) -> Alive(x)",
            [[
                literal(True, "NotKilled", "?x"),
                literal(False, "Alive", "?x")
            ]]
        )
    match = re.fullmatch(r"(\w+) is alive", s)
    if match:
        return (
            f"Alive({match.group(1)})",
            [[literal(False, "Alive", match.group(1))]]
        )
    match = re.fullmatch(r"(\w+) is food", s)
    if match:
        return (
            f"Food({match.group(1)})",
            [[literal(False, "Food", match.group(1))]]
        )
    match = re.fullmatch(r"(\w+) eats (\w+)", s)
    if match:
        a, b = match.groups()
        return (
            f"Eats({a},{b})",
            [[literal(False, "Eats", a, b)]]
        )
    raise ValueError("Sentence pattern not supported: " + sentence)

def convert_query(query):
    match = re.fullmatch(
        r"\s*(\w+) likes (\w+)\.?\s*",
        query.lower()
    )
    if match:
        a, b = match.groups()
        return literal(False, "Likes", a, b)
    raise ValueError("Query format should be: John likes peanuts")

def apply_substitution(lit, subs):
    neg, pred, args = lit
    new_args = []
    for arg in args:
        while arg in subs:
            arg = subs[arg]
        new_args.append(arg)
    return (neg, pred, tuple(new_args))

def resolve(c1, c2):
    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):
            neg1, pred1, args1 = l1
            neg2, pred2, args2 = l2
            if pred1 != pred2 or neg1 == neg2:
                continue
            if len(args1) != len(args2):
                continue
            subs = {}
            for a, b in zip(args1, args2):
                result = unify(a, b, subs)
                if result is None:
                    break
            else:
                new_clause = []
                for lit in c1[:i] + c1[i+1:] + c2[:j] + c2[j+1:]:
                    new_clause.append(
                        apply_substitution(lit, subs)
                    )
                new_clause = list(dict.fromkeys(new_clause))
                return new_clause
    return None

def resolution_proof(clauses):
    proof_tree = []
    for i, clause in enumerate(clauses):
        proof_tree.append({
            "clause": clause,
            "parent": None
        })
    known = [tuple(map(tuple, c)) for c in clauses]
    step = 1
    while step <= 1000:
        new_found = False
        for i in range(len(proof_tree)):
            for j in range(i + 1, len(proof_tree)):
                c1 = proof_tree[i]["clause"]
                c2 = proof_tree[j]["clause"]
                result = resolve(c1, c2)
                if result is None:
                    continue
                key = tuple(result)
                if key in known:
                    continue
                known.append(key)
                proof_tree.append({
                    "clause": result,
                    "parent": (i, j)
                })
                new_index = len(proof_tree) - 1
                step += 1
                new_found = True
                if len(result) == 0:
                    return proof_tree, new_index, True
        if not new_found:
            break
    return proof_tree, None, False

def print_proof_tree(tree, index, indent=""):
    node = tree[index]
    clause_str = display_clause(node["clause"])
    if node["parent"] is None:
        print(f"{indent}Leaf -> C{index+1}: {clause_str}")
    else:
        p1, p2 = node["parent"]
        def contains_index(t, idx, target):
            if idx == target:
                return True
            n = t[idx]
            if n["parent"] is not None:
                return contains_index(t, n["parent"][0], target) or contains_index(t, n["parent"][1], target)
            return False

        # Force the branch containing the negated query (index 9) to print first
        if contains_index(tree, p1, 9):
            first_branch, second_branch = p1, p2
        else:
            first_branch, second_branch = p2, p1

        print_proof_tree(tree, first_branch, indent + "  ├─ ")
        print_proof_tree(tree, second_branch, indent + "  └─ ")
        print(f"{indent}C{index+1}: {clause_str} (from C{p1+1} & C{p2+1})")

knowledge_base = [
    "John likes all kind of food",
    "Apple and vegetables are food",
    "Anything anyone eats and not killed is food",
    "Anil eats peanuts and still alive",
    "Harry eats everything that Anil eats",
    "Anyone who is alive implies not killed",
    "Anyone who is not killed implies alive"
]
query = "John likes peanuts"

clauses = []
for sentence in knowledge_base:
    _, cnf = convert_to_fol(sentence)
    clauses.extend(cnf)

query_literal = convert_query(query)
negated_query = (not query_literal[0], query_literal[1], query_literal[2])
clauses.append([negated_query])

tree, final_index, proved = resolution_proof(clauses)

if proved:
    print("PROOFTREE SUCCESS:")
    print_proof_tree(tree, final_index)
else:
    print("Failed to prove.")

PROOFTREE SUCCESS:
  ├─   ├─   ├─   ├─   ├─ Leaf -> C10: ~Likes(john, peanuts)
  ├─   ├─   ├─   ├─   └─ Leaf -> C1: ~Food(?x) OR Likes(john, ?x)
  ├─   ├─   ├─   ├─ C14: ~Food(peanuts) (from C1 & C10)
  ├─   ├─   ├─   └─ Leaf -> C4: ~Eats(?y, ?x) OR ~NotKilled(?y) OR Food(?x)
  ├─   ├─   ├─ C18: ~Eats(?y, peanuts) OR ~NotKilled(?y) (from C4 & C14)
  ├─   ├─   └─ Leaf -> C5: Eats(anil, peanuts)
  ├─   ├─ C23: ~NotKilled(anil) (from C5 & C18)
  ├─   └─ Leaf -> C8: ~Alive(?x) OR NotKilled(?x)
  ├─ C36: ~Alive(anil) (from C8 & C23)
  └─ Leaf -> C6: Alive(anil)
C73: {} (from C6 & C36)
